# F2.1 · Obtención de los datos

De dónde vienen los datos, cómo se descargaron y cómo se comprueba que nadie los modificó.

In [1]:
# Configuración común: rutas del proyecto y módulos del pipeline.
import pandas as pd

from pipeline import chilecompra as cc
from pipeline import config, ordenes

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.1f}".format)
print("Raíz del proyecto:", config.PROJECT_ROOT.name)

Raíz del proyecto: 002_sumativa-1-avance-proyecto-f1-y-f2-40


## 1. Fuente y licencia

Usamos los datos abiertos de **ChileCompra** (Dirección de Compras y Contratación Pública): un archivo mensual de **órdenes de compra** y otro de **licitaciones**. Los términos de uso permiten reutilizarlos citando la fuente.

El estudio compara **septiembre de 2024** con **septiembre de 2025**: el mismo mes, antes y después de los cambios de la Ley N.° 21.634 (vigentes desde diciembre de 2024).

In [2]:
for source in cc.SOURCES.values():
    for month in cc.DEFAULT_MONTHS:
        year, number = cc.parse_month(month)
        print(cc.source_url(source, year, number))
print("Términos de uso:", cc.TERMS_URL)

https://transparenciachc.blob.core.windows.net/oc-da/2024-9.zip
https://transparenciachc.blob.core.windows.net/oc-da/2025-9.zip
https://transparenciachc.blob.core.windows.net/lic-da/2024-9.zip
https://transparenciachc.blob.core.windows.net/lic-da/2025-9.zip
Términos de uso: https://www.chilecompra.cl/terminos-y-condiciones-de-uso/


## 2. Descarga y filtro reproducibles

Cada archivo nacional pesa cerca de 100 MB, así que `scripts/fetch_data.py` lo descarga, conserva solo el sector **Salud** de la **Región de Valparaíso** y guarda ese subconjunto en `data/raw/`.

**Decisión:** el subconjunto se versiona y no se vuelve a descargar en cada ejecución, porque ChileCompra regenera sus archivos mensuales y una descarga nueva podría traer datos distintos.

```bash
python scripts/fetch_data.py            # descarga y filtra
python scripts/fetch_data.py --verify   # solo comprueba las huellas
```

## 3. Carga y verificación de integridad

El manifiesto guarda la huella SHA-256 de cada archivo. Si un archivo cambió, la verificación falla y el notebook se detiene.

In [3]:
manifest_path = config.RAW_DATA_DIR / "manifest.json"
checks = cc.verify_manifest(config.RAW_DATA_DIR, manifest_path)
for name, ok in checks.items():
    print("OK   " if ok else "FALLA", name)
assert all(checks.values()), "Un archivo de data/raw/ no coincide con su huella"

OK    lic_2024-09_salud_valparaiso.csv
OK    lic_2025-09_salud_valparaiso.csv
OK    oc_2024-09_salud_valparaiso.csv
OK    oc_2025-09_salud_valparaiso.csv


In [4]:
manifest = cc.load_manifest(manifest_path)["files"]
summary = pd.DataFrame(manifest).T[["dataset", "national_rows", "rows", "columns", "downloaded_at"]]
summary.columns = ["conjunto", "filas del archivo nacional", "filas del subconjunto", "columnas", "descargado"]
summary

,conjunto,filas del archivo nacional,filas del subconjunto,columnas,descargado
lic_2024-09_salud_valparaiso.csv,Licitaciones con ofertas,178603,2726,110,2026-11-08T18:41:54+00:00
lic_2025-09_salud_valparaiso.csv,Licitaciones con ofertas,124680,2361,110,2026-11-08T18:43:24+00:00
oc_2024-09_salud_valparaiso.csv,Órdenes de compra,396516,12025,78,2026-11-08T18:52:46+00:00
oc_2025-09_salud_valparaiso.csv,Órdenes de compra,407358,13855,79,2026-11-08T19:02:53+00:00


## 4. Diccionario de datos inicial

Variables de las órdenes de compra que usa el estudio. Cada fila del archivo es una **línea de producto**: una orden con tres productos ocupa tres filas.

In [5]:
raw = ordenes.load_orders(config.RAW_DATA_DIR)
print(f"{len(raw):,} líneas, {raw['Codigo'].nunique():,} órdenes, {raw.shape[1]} columnas")

dictionary = pd.DataFrame(
    [
        ("Codigo", "identificador", "Código de la orden de compra"),
        ("FechaEnvio", "fecha", "Día en que la orden se envió al proveedor"),
        ("EsTratoDirecto", "binaria", "La orden proviene de un trato directo"),
        ("EsCompraAgil", "binaria", "La orden proviene de una compra ágil"),
        ("Tipo", "nominal", "Tipo de orden (AG, SE, CM, TD, CC)"),
        ("CodigoLicitacion", "identificador", "Licitación de origen, si existe"),
        ("TipoMonedaOC", "nominal", "Moneda de la orden (CLP, CLF, USD, UTM)"),
        ("MontoTotalOC_PesosChilenos", "continua", "Monto total de la orden en pesos"),
        ("UnidadCompra", "nominal", "Unidad que compra (hospital, consultorio, etc.)"),
        ("NombreProveedor", "nominal", "Proveedor adjudicado"),
    ],
    columns=["variable", "rol", "descripción"],
)
dictionary["ejemplo"] = [raw[v].iloc[0] for v in dictionary["variable"]]
dictionary

25,880 líneas, 10,285 órdenes, 80 columnas


,variable,rol,descripción,ejemplo
0,Codigo,identificador,Código de la orden de compra,2200-524-SE23
1,FechaEnvio,fecha,Día en que la orden se envió al proveedor,2024-09-27
2,EsTratoDirecto,binaria,La orden proviene de un trato directo,Si
3,EsCompraAgil,binaria,La orden proviene de una compra ágil,No
4,Tipo,nominal,"Tipo de orden (AG, SE, CM, TD, CC)",SE
5,CodigoLicitacion,identificador,"Licitación de origen, si existe",
6,TipoMonedaOC,nominal,"Moneda de la orden (CLP, CLF, USD, UTM)",CLP
7,MontoTotalOC_PesosChilenos,continua,Monto total de la orden en pesos,"3401030,71"
8,UnidadCompra,nominal,"Unidad que compra (hospital, consultorio, etc.)",Servicio de Salud Aconcagua
9,NombreProveedor,nominal,Proveedor adjudicado,CHILQUINTA DISTRIBUCIÓN S.A.


El perfil completo de las 79 variables, generado con el validador del curso, está en `docs/validacion/`.